# Prefix Length vs Chain-of-Thought Analysis

Loads JSONL results produced by `scripts/run_experiment.py` and plots breadth, depth, warnings, and answer accuracy versus prefix length.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

RESULTS_DIR = Path('../results')

def load_jsonl(path: Path):
    rows = []
    if not path.exists():
        return pd.DataFrame()
    with path.open() as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return pd.DataFrame(rows)

frames = []
for path in sorted(RESULTS_DIR.glob('*.jsonl')):
    df = load_jsonl(path)
    if not df.empty:
        df['model_id'] = path.stem
        frames.append(df)

if not frames:
    raise SystemExit('No results found in results/*.jsonl')

data = pd.concat(frames, ignore_index=True)
data['max_depth'] = data['tree_metrics'].map(lambda x: x['max_depth'])
data['total_nodes'] = data['tree_metrics'].map(lambda x: x['total_nodes'])
data['breadth_warning_count'] = data['tree_metrics'].map(lambda x: x['breadth_warning_count'])
data['answer_correct'] = data['trace_metrics'].map(lambda x: x['answer_correct'])
data['top_k_any_correct'] = data['top_k_metrics'].map(lambda x: x['top_k_any_correct'])
data['mean_entropy_reasoning'] = data['trace_metrics'].map(lambda x: x.get('mean_entropy_reasoning'))
data.head()

In [ ]:
summary = (
    data.groupby(['model_id', 'prefix_length'])
    .agg(
        max_depth=('max_depth', 'mean'),
        total_nodes=('total_nodes', 'mean'),
        breadth_warnings=('breadth_warning_count', 'sum'),
        greedy_accuracy=('answer_correct', 'mean'),
        top_k_accuracy=('top_k_any_correct', 'mean'),
        mean_entropy=('mean_entropy_reasoning', 'mean'),
    )
    .reset_index()
)
summary

In [ ]:
for model_id, group in summary.groupby('model_id'):
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    fig.suptitle(model_id)
    axes[0, 0].plot(group['prefix_length'], group['max_depth'], marker='o')
    axes[0, 0].set_title('Max depth vs prefix length')
    axes[0, 1].plot(group['prefix_length'], group['total_nodes'], marker='o')
    axes[0, 1].set_title('Total nodes vs prefix length')
    axes[1, 0].plot(group['prefix_length'], group['breadth_warnings'], marker='o')
    axes[1, 0].set_title('Breadth warnings vs prefix length')
    axes[1, 1].plot(group['prefix_length'], group['greedy_accuracy'], marker='o', label='greedy')
    axes[1, 1].plot(group['prefix_length'], group['top_k_accuracy'], marker='o', label='top-k')
    axes[1, 1].set_title('Accuracy vs prefix length')
    axes[1, 1].legend()
    for ax in axes.flat:
        ax.set_xlabel('prefix length')
    plt.tight_layout()
    plt.show()